# 623 SPP v18: hard distinct-action feedback LSTM

The lossless line58 + callback-kind encoder, chronological `DEMAND(addr)` / `CACHE_FILL(evicted_addr)` inputs, labels, splits, and one global LSTM are unchanged. v18 removes count sampling, hard-quantizes a scale-aware distinct non-self delta, draws fill directly from the learned posterior with a float64 keyed uniform, and feeds the actual hard delta/fill action to the next rank through straight-through training values. Teacher count schedules training loss ranks only. PC, private SPP state, thresholds, budgets, candidates, page rules, and future rows remain excluded. This is matched-input open-loop replay, not a closed-loop live-NN claim.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID='623_offline_lstm_spp_hard_distinct_v18_seed7'
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_spp/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified',archive)


In [ ]:
import json
TRACE='623.xalancbmk_s-700B'; POLICY='spp'; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
 for path in items.values(): assert os.path.isfile(path),path
historical_manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected_input_provenance={'status':'PASS','experiment_revision':'spp_source_input_variable_delta_fill_feedback_free_running_v11','event_logger_schema':'623_causal_trigger_fill_v6','neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr'],'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'fill_lead_cutoff_used':False,'inference_policy_hardcodes_used':False}
bad={k:(historical_manifest.get(k),v) for k,v in expected_input_provenance.items() if historical_manifest.get(k)!=v}; assert not bad,bad
assert historical_manifest['training_runtime_fields']==['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr']==historical_manifest['inference_runtime_fields']
COLLECTION_MANIFEST_ROLE='historical_input_package_provenance_only'
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/train_and_offline_infer.py'
SOURCE=f'{INPUT_DIR}/spp_source_contract.json'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
POINTS=[(8,'p0',2664),(16,'p1',6208),(32,'p2',15984),(64,'p3',46288),(128,'p4',149904)]
SPECS=[{'tag':f'hard_distinct_delta_fill_spp_lstm_h{size}','family':'lstm','size':size,'pair':pair,'parameters':parameters} for size,pair,parameters in POINTS]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-teacher-actions',INPUTS[role]['teacher']]
 cmd += ['--source-contract',SOURCE,'--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--decoder-seed','7','--epochs','10','--chunk-len','1024','--accumulate-chunks','16']
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={
  'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm',
  'operation':'train-v18','model_revision':'compact_crn_hard_distinct_delta_keyed_fill_v18','decoder_revision':'hard_distinct_delta_keyed_fill_v18',
  'parameter_count':spec['parameters'],'runtime_feature_count':59,
  'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,
  'training_inference_input_encoder_identical':True,
  'decoder_training_mode':'teacher_count_scheduled_loss_with_hard_self_action_feedback',
  'decoder_previous_teacher_action_used_as_input':False,
  'decoder_free_running_self_test':'PASS',
  'model_input_is_causal_external_event_sequence_only':True,
  'cache_fill_feedback_used_as_raw_external_input':True,'model_does_not_use_pc':True,
  'normal_policy_outputs_used_as_model_inputs':False,
  'normal_policy_outputs_used_as_training_targets':True,
  'probability_threshold_used':False,'neural_degree_cap':None,
  'gate_class_weighting_used':False,'gate_training_objective':'unweighted_bernoulli_nll',
  'gate_decoding_rule':'deterministic_raw_logit_sign',
  'request_count_training_objective':'unweighted_bernoulli_hurdle_plus_positive_poisson_excess_nll',
  'request_count_decoding_rule':'deterministic_raw_hurdle_plus_rounded_conditional_excess_mean',
  'request_count_residual_scope':'none_event_local',
  'common_random_numbers_across_capacities':True,
  'strict_common_random_numbers_across_capacities':True,
  'cross_event_rng_state_used':False,'stochastic_decoding_reproducible':True,
  'decoder_sampling_roles':['train','eval'],'decoder_train_sampling_performed':True,'decoder_count_sampling_performed':False,'joint_delta_fill_dependency_modeled':False,
  'joint_pair_classes':0,'joint_delta_fill_training_objective':None,
  'joint_delta_fill_decoding_rule':None,'delta_mixture_components':4,
  'delta_training_objective':'four_component_signed_log_delta_mixture_nll',
  'delta_mixture_decoding_rule':'component_peak_density_order_then_hard_quantized_legal_delta',
  'fill_training_objective':'unweighted_two_class_cross_entropy',
  'fill_decoding_rule':'event_keyed_categorical_inverse_cdf','fill_argmax_used':False,
  'fill_probability_feedback_used':False,'hard_fill_one_hot_feedback_used':True,'keyed_fill_uniform_dtype':'float64',
  'delta_decoder_feedback_rule':'actual_hard_quantized_emitted_delta_with_straight_through_training',
  'sampled_outputs_used_as_decoder_feedback':True,'address_confidence_fill_heuristic_used':False,
  'guard_selected_decoder':False,'joint_map_used':False,'weights_retrained':True,
  'same_source_input_offline_claim_allowed':True,'closed_loop_live_claim_allowed':False,
  'keyed_sampling_self_test':'PASS','deterministic_hurdle_count_self_test':'PASS','hard_distinct_action_feedback_self_test':'PASS','factorized_delta_fill_sampling_self_test':'PASS',
  'experiment_revision':'spp_source_input_variable_delta_fill_feedback_free_running_v11'
 }
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['training_runtime_fields']==['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr']==meta['inference_runtime_fields']
 encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
 assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
 fill_counts=meta.get('offline_nn_fill_level_counts',{})
 assert sum(fill_counts.values())==meta['offline_nn_entries'],(fill_counts,meta['offline_nn_entries'])
 assert meta['offline_nn_entries']==meta['materialized_distinct_action_count']==meta['raw_predicted_action_count']
 legality=meta['action_legality_diagnostics']; assert legality['self_target_actions']==0 and legality['duplicate_target_actions']==0,legality
 assert meta['offline_nn_entries']==0 or fill_counts.get('FILL_L2',0)>0,fill_counts
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','decision_rule','offline_normal_entries','offline_nn_entries','offline_nn_fill_level_counts','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':'spp_source_input_variable_delta_fill_feedback_free_running_v11','model_revision':'compact_crn_hard_distinct_delta_keyed_fill_v18','collection_manifest_role':COLLECTION_MANIFEST_ROLE,'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))


In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the v18 output archive to the matching server run and launch replay. Teacher actions remain loss/comparator data only; each next-rank feedback value is the model's own materialized hard delta and keyed hard fill.